# Power consumption analysis of SQD
This Jupyter notebook prints the Qiskit Runtime usage and the power consumption of the classical processing of an SQD job.
It prompts for a `job_id` (as a character string) and for the CSV file returned by the eco2AI tracker in the `SQD_Alain` class.
|||
|-|-|
|**Author:** |Alain Chancé|
|**Date:** |December 11, 2025|
|**Version:** |**1.00**<br/>*Details see at the end of this notebook*|
<br/>

# MIT License

Copyright (c) 2025 Alain Chancé

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

## Import required dependencies

In [1]:
import os
import pandas as pd
import re

from qiskit_ibm_runtime import QiskitRuntimeService

## Prompt for a job_id

In [2]:
#--------------------------------------------------------------------------------------------
# Prompt for a job_id as a string (letters and digits only, no spaces or special characters)
#--------------------------------------------------------------------------------------------
def get_valid_string():
    while True:
        user_input = input("Enter job_id as a string (letters and digits only, no spaces or special characters): ")

        # Regex: only allows A-Z, a-z, 0-9
        if re.fullmatch(r"[A-Za-z0-9]+", user_input):
            print("Valid input job_id:", user_input)
            return user_input
        else:
            print("❌ Invalid input. Please try again.")

In [3]:
job_id = get_valid_string()

Enter job_id as a string (letters and digits only, no spaces or special characters):  d4utbmkgk3fc73aubgag


Valid input job_id: d4utbmkgk3fc73aubgag


In [4]:
service = QiskitRuntimeService()
job = service.job(job_id)

metrics = job.metrics()           # Fetch metrics from server
usage = metrics.get("usage", {})
QPU_usage = usage.get("quantum_seconds")
print(f"Qiskit Runtime usage: {QPU_usage:.2f} seconds")

Qiskit Runtime usage: 21.00 seconds


## Rough estimation of the energy consumption on the IBM Quantum device
**Assumption**
A ballpark figure for a typical modern IBM-class superconducting quantum computer (including cryogenics + support, while idle or lightly used): ~ 15–25 kW. Source: [Green quantum computing, Capgemini, 8 May 2023](https://www.capgemini.com/insights/expert-perspectives/green-quantum-computing/).

In [5]:
power_QPU = 25.00
print("A rough estimate of the energy consumption is computed as power_QPU (kW) * QPU_usage (s) * 1/3600 (h/s)")
print(f"power_QPU (kW): {power_QPU}")
print(f"QPU_usage/3600 (h): {QPU_usage/3600:.4f}")
print(f"Rough estimate of the power consumption on the IBM QPU (kWh) = {power_QPU*QPU_usage/3600:.4f}")

A rough estimate of the energy consumption is computed as power_QPU (kW) * QPU_usage (s) * 1/3600 (h/s)
power_QPU (kW): 25.0
QPU_usage/3600 (h): 0.0058
Rough estimate of the power consumption on the IBM QPU (kWh) = 0.1458


## Prompt for a eco2AI filename

In [6]:
#------------------------------
# Prompt for a eco2AI filename
#------------------------------
eco2AI_file = "SQD.csv"
while True:
    eco2AI_file = input("Enter eco2AI filename (.csv): ").strip()
    if eco2AI_file.lower().endswith((".csv")):
        print("You entered: ", eco2AI_file)
        if os.path.isfile(eco2AI_file):
            break
        else:
            print("Missing eco2AI file: ", eco2AI_file)    
    else:
        print("Invalid file type. Please enter a .csv file.")

Enter eco2AI filename (.csv):  SQD_CO2_ibm_fez.csv


You entered:  SQD_CO2_ibm_fez.csv


## Read the CSV file and print the power consumption of classical processing

In [7]:
# Read the CSV file
df = pd.read_csv(eco2AI_file, sep =',')

# Access column by name
try:
    power_usage = df["power_consumption(kWh)"].iloc[0]
    print(f"Power consumption of classical processing (kWh): {power_usage:.4f}")
except Exception as e:
    print(f"Error accessing the eco2AI file: {e}")

Power consumption of classical processing (kWh): 0.0174
